In [ ]:
import sys; sys.path.append("..")
import math
import numpy as np
from IPython.display import Image, display
from common.gifutil import ascii_frames_to_gif

In [ ]:
W, H = 80, 40
CHARS = np.array(list(".,-~:;=!*#$@"))

R1 = 2
R2 = 4
K2 = 8
K1_X, K1_Y = 30, 15
CX, CY = W/2, H/2

theta, phi = np.meshgrid(np.arange(0, 628, 7) / 100,
                         np.arange(0, 628, 2) / 100,
                         indexing="ij")
theta, phi = theta.ravel(), phi.ravel()
sin_theta, cos_theta = np.sin(theta), np.cos(theta)
sin_phi, cos_phi = np.sin(phi), np.cos(phi)

radius = R2 + R1 * cos_theta

def render_frame(t: float) -> str:
    A = t
    B = t * 0.5
    sin_A, cos_A = math.sin(A), math.cos(A)
    sin_B, cos_B = math.sin(B), math.cos(B)

    y_a = radius * sin_phi * cos_A - R1 * sin_theta * sin_A
    z = K2 + radius * sin_phi * sin_A + R1 * sin_theta * cos_A
    inv_z = 1 / z
    x_rot = radius * cos_phi * cos_B - y_a * sin_B
    y_rot = radius * cos_phi * sin_B + y_a * cos_B

    col = (CX + K1_X * inv_z * x_rot).astype(int)
    row = (CY + K1_Y * inv_z * y_rot).astype(int)

    lum = (8 * ((sin_theta * sin_A - sin_phi * cos_theta * cos_A) * cos_B
                - sin_phi * cos_theta * sin_A
                - sin_theta * cos_A
                - cos_phi * cos_theta * sin_B)).astype(int)
    lum = np.clip(lum, 0, len(CHARS) - 1)

    ok = (row > 0) & (row < H) & (col > 0) & (col < W)
    idx, inv_z, lum = (col + W * row)[ok], inv_z[ok], lum[ok]

    order = np.argsort(inv_z[::-1], kind="stable")
    idx, lum = idx[::-1][order], lum[::-1][order]

    flat = np.full(W * H, " ")
    flat[idx] = CHARS[lum]
    return "\n".join("".join(r) for r in flat.reshape(H, W))

frames = [render_frame(i * 4 * math.pi / 120) for i in range(120)]

In [ ]:
ascii_frames_to_gif(frames, "out/demo.gif", fps=30)
display(Image("out/demo.gif"))